In [9]:
get_ipython().system('pip install ucimlrepo')
from ucimlrepo import fetch_ucirepo
abalone_ds = fetch_ucirepo(id=1)
features = abalone_ds.data.features
targets = abalone_ds.data.targets

In [10]:
import pandas as pd
df = pd.concat([features, targets], axis=1)
df.shape[0]


4177

In [11]:
list(df.columns)


['Sex',
 'Length',
 'Diameter',
 'Height',
 'Whole_weight',
 'Shucked_weight',
 'Viscera_weight',
 'Shell_weight',
 'Rings']

In [12]:
df.head()


,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [13]:
# What serves as input here?
# Every column except the last one acts as an input feature. The final column is what we want to predict.

# What is the expected output?
# The model should output a single numeric value -- the predicted ring count of the abalone.

# Why is the output continuous rather than categorical?
# Ring count represents a measurable quantity along a continuous scale, making this a regression task.


In [14]:
targets = targets + 1.5


In [15]:
features = features[["Length", "Diameter", "Shell_weight"]]
features.head()


,Length,Diameter,Shell_weight
0,0.455,0.365,0.150
1,0.350,0.265,0.070
2,0.530,0.420,0.210
3,0.440,0.365,0.155
4,0.330,0.255,0.055


In [16]:
# Why these three features?

# Feature chosen -- Length:
# Reflects the overall physical size of the abalone.
# Larger animals tend to be older, so length acts as a proxy for age.

# Feature chosen -- Diameter:
# Complements Length by capturing the width dimension.
# Together they approximate the body volume more accurately.

# Feature chosen -- Shell_weight:
# Shell mass accumulates over time as the animal grows.
# It provides a direct physical signal of biological age.


In [17]:
import numpy as np
X_arr = features.values
Y_arr = targets.values

total = X_arr.shape[0]
cut = int(0.8 * total)

X_train = X_arr[:cut]
X_test  = X_arr[cut:]
y_train = Y_arr[:cut]
y_test  = Y_arr[cut:]

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


X_train shape: (3341, 3)
X_test shape : (836, 3)
y_train shape: (3341, 1)
y_test shape : (836, 1)


In [18]:
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)
X_train = (X_train - mu) / sigma
X_test  = (X_test  - mu) / sigma


In [19]:
# Why do we normalize before training?
# The raw features live on very different numerical scales -- shell weight and length differ by orders of magnitude.
# When scales vary, the gradient descent update is pulled much more strongly by whichever feature has the largest range.
# Standardizing each feature to zero mean and unit variance keeps gradient steps balanced and helps the model converge faster.


In [20]:
def predict_linear(X, w, b):
    y_pred = X @ w + b
    print("Shape X:", X.shape)
    print("Shape w:", w.shape)
    print("Shape b:", np.shape(b))
    print("Shape y_pred:", y_pred.shape)
    return y_pred

# Note:
# Learnable parameters are the weight vector w and the scalar bias b.
# Total parameter count = d weights + 1 bias, where d is the number of input features.


In [21]:
def compute_mse(y_true, y_pred):
    loss = np.mean((y_true - y_pred) ** 2)
    return loss

# Notes:
# Squaring errors keeps everything positive and penalises large mistakes disproportionately.
# This means the model is incentivised to avoid catastrophic predictions.


In [22]:
# Gradient intuition:
# The gradient tells us how steeply the loss changes when we nudge each parameter.
# Subtracting the gradient at each step moves the parameters in the direction that reduces the loss.


In [23]:
def compute_grad_w(X, y_true, y_pred):
    N = len(y_true)
    dW = (2/N) * X.T @ (y_pred - y_true)
    return dW


def compute_grad_b(y_true, y_pred):
    N = len(y_true)
    db = (2/N) * np.sum(y_pred - y_true)
    return db

# Notes:
# A very large gradient value signals the loss surface is steep -- small weight changes have a big effect.
# A learning rate that is too high can overshoot the minimum, causing the loss to oscillate or blow up.


In [24]:
num_features = X_train.shape[1]
w = np.random.randn(num_features, 1) * 0.01
b = 0

lr = 0.01
num_epochs = 500

for ep in range(num_epochs):
    y_pred = X_train @ w + b
    loss = compute_mse(y_train, y_pred)
    dW = compute_grad_w(X_train, y_train, y_pred)
    db = compute_grad_b(y_train, y_pred)
    w = w - lr * dW
    b = b - lr * db

    if ep % 50 == 0:
        print("Epoch:", ep, "Loss:", loss)


# Before training:
# Expected the loss to steadily decrease epoch by epoch.

# After training:
# Loss decreased consistently -- confirming the learning rate and setup were appropriate.


Epoch: 0 Loss: 144.3520511551562
Epoch: 50 Loss: 24.608831028651313
Epoch: 100 Loss: 9.199383838800104
Epoch: 150 Loss: 7.106684310587452
Epoch: 200 Loss: 6.7898342196440025
Epoch: 250 Loss: 6.717186842445727
Epoch: 300 Loss: 6.683630196846639
Epoch: 350 Loss: 6.660429798071405
Epoch: 400 Loss: 6.642594345849642
Epoch: 450 Loss: 6.62855251093189


In [25]:
y_test_pred = X_test @ w + b

test_mse = compute_mse(y_test, y_test_pred)
test_mae = np.mean(np.abs(y_test - y_test_pred))

print("Test MSE:", test_mse)
print("Test MAE:", test_mae)

print("\nSample Predictions:\n")

for i in range(5):
    actual = y_test[i][0]
    predicted = y_test_pred[i][0]
    err = abs(actual - predicted)

    print("True Age:", actual,
          " Predicted:", predicted,
          " Absolute Error:", err)

# Observations:
# The model struggles most on older specimens where ring count is higher.
# Predictions tend to cluster near the dataset mean -- linear regression is pulled toward minimising
# squared error globally, which biases it toward central values rather than extremes.


Test MSE: 5.0943772248378645
Test MAE: 1.7324554640077285

Sample Predictions:

True Age: 13.5  Predicted: 10.816600681719772  Absolute Error: 2.6833993182802285
True Age: 15.5  Predicted: 9.695056143571433  Absolute Error: 5.804943856428567
True Age: 14.5  Predicted: 10.10998944813962  Absolute Error: 4.39001055186038
True Age: 14.5  Predicted: 11.112130504673928  Absolute Error: 3.387869495326072
True Age: 13.5  Predicted: 11.348330549460456  Absolute Error: 2.1516694505395435
